## DATA VALIDATION

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
ad_spend = pd.read_csv(r"C:\Users\Yusuf\OneDrive\Desktop\cleaned_ad_spend.csv")
touchpoints = pd.read_csv(r"C:\Users\Yusuf\OneDrive\Desktop\cleaned_user_touchpoints.csv")
conversions = pd.read_csv(r"C:\Users\Yusuf\OneDrive\Desktop\cleaned_conversions.csv")

C:\Users\Yusuf\AppData\Local\Temp\ipykernel_12388\2819352979.py:2: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  touchpoints = pd.read_csv(r"C:\Users\Yusuf\OneDrive\Desktop\cleaned_user_touchpoints.csv")


In [3]:
print(ad_spend.shape)

(750, 8)


In [4]:
print(touchpoints.shape)

(75000, 12)


In [5]:
print(conversions.shape)

(7500, 8)


### User-ID Validation

In [6]:
print("Unique users in touchpoints:", touchpoints["user_id"].nunique())

Unique users in touchpoints: 18000


In [7]:
print("Unique users in conversions:", conversions["user_id"].nunique())

Unique users in conversions: 7500


In [8]:
touch_users = set(touchpoints["user_id"])
conv_users = set(conversions["user_id"])

touch_only = touch_users - conv_users
conv_only = conv_users - touch_users

print("Users with touchpoints but no conversion:",
      len(touch_only))

print("Users with conversion but no touchpoints:",
      len(conv_only))

Users with touchpoints but no conversion: 10500
Users with conversion but no touchpoints: 0


### Campaign Validation

In [9]:
print("Campaigns in touchpoints:",
      touchpoints["campaign_id"].nunique(dropna=True))

print("Campaigns in ad_spend:",
      ad_spend["campaign_id"].nunique())

Campaigns in touchpoints: 27
Campaigns in ad_spend: 27


In [10]:
tp_campaigns = set(
    touchpoints["campaign_id"].dropna()
)

spend_campaigns = set(
    ad_spend["campaign_id"]
)

print("Campaigns in touchpoints but not spend:",
      len(tp_campaigns - spend_campaigns))

print("Campaigns in spend but not touchpoints:",
      len(spend_campaigns - tp_campaigns))

Campaigns in touchpoints but not spend: 0
Campaigns in spend but not touchpoints: 0


### User Journey & Funnel Exploration

In [11]:
touchpoints_per_user = (
    touchpoints
    .groupby("user_id")
    .size()
)

print(touchpoints_per_user.describe())

count    18000.000000
mean         4.166667
std         70.512546
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max       9460.000000
dtype: float64


In [12]:
journey_distribution = (
    touchpoints
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

print(journey_distribution.head(20))

1       2671
2       3491
3       3504
4       2749
5       2180
6       1459
7       1112
8        714
9         43
10        28
11        26
12        13
13         8
16         1
9460       1
Name: count, dtype: int64


In [13]:
avg_touchpoints = (
    touchpoints
    .groupby("user_id")
    .size()
    .mean()
)

print("Average touchpoints per user:",
      round(avg_touchpoints,2))

Average touchpoints per user: 4.17


In [14]:
touchpoints_per_user = (
    touchpoints.groupby("user_id")
    .size()
    .reset_index(name="touchpoint_count")
)

touchpoints_per_user.sort_values(
    "touchpoint_count",
    ascending=False
).head(10)

,user_id,touchpoint_count
8214,USR0008215,9460
16511,USR0016512,16
10506,USR0010507,13
15740,USR0015741,13
3504,USR0003505,13
16135,USR0016136,13
11337,USR0011338,13
9713,USR0009714,13
9224,USR0009225,13
2032,USR0002033,13


In [15]:
touchpoints_per_user = (
    touchpoints.groupby("user_id")
    .size()
)

print("Mean including outlier:",
      touchpoints_per_user.mean())

print("Median:",
      touchpoints_per_user.median())

print("95th percentile:",
      touchpoints_per_user.quantile(0.95))

print("99th percentile:",
      touchpoints_per_user.quantile(0.99))

Mean including outlier: 4.166666666666667
Median: 3.0
95th percentile: 7.0
99th percentile: 8.0


In [16]:
filtered = touchpoints_per_user[
    touchpoints_per_user < 100
]

print("Mean after removing outlier:",
      filtered.mean())

Mean after removing outlier: 3.64131340630035


#### One user (USR0008215) generated 9,460 touchpoints, while the 99th percentile of users had only 8 touchpoints. This record was identified as an extreme outlier and excluded from journey-length analysis.

In [17]:
outlier_user = "USR0008215"
print(outlier_user)

USR0008215


### Conversion Journey Analysis

In [18]:
# Count Touchpoints for Converting Users
converter_touchpoints = (
    touchpoints[
        touchpoints["user_id"].isin(
            conversions["user_id"]
        )
    ]
    .groupby("user_id")
    .size()
)

converter_touchpoints.describe()

count    7500.000000
mean        3.633333
std         2.027273
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        13.000000
dtype: float64

In [19]:
single_touch = (converter_touchpoints == 1).sum()

multi_touch = (converter_touchpoints > 1).sum()

print("Single-touch converters:", single_touch)
print("Multi-touch converters:", multi_touch)

Single-touch converters: 1118
Multi-touch converters: 6382


In [20]:
total = len(converter_touchpoints)

print(
    "Single-touch %",
    round(single_touch / total * 100, 2)
)

print(
    "Multi-touch %",
    round(multi_touch / total * 100, 2)
)

Single-touch % 14.91
Multi-touch % 85.09


In [21]:
total_users = touchpoints["user_id"].nunique()

converted_users = conversions["user_id"].nunique()

conversion_rate = (
    converted_users / total_users
) * 100

print(f"Conversion Rate: {conversion_rate:.2f}%")

Conversion Rate: 41.67%


In [22]:
touchpoints['source'].value_counts()

source
google      18216
referral     8719
bing         8667
direct       8662
blog         8630
facebook     6976
email        5825
linkedin     4682
tiktok       4623
Name: count, dtype: int64

In [23]:
touchpoints['medium'].value_counts()

medium
organic        17435
paid_social    16281
cpc             9448
referral        8719
none            8662
content         8630
email           5825
Name: count, dtype: int64

In [24]:
# How many conversions per user?

conversions.groupby("user_id").size().describe()

count    7500.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64

In [25]:
multi_conversion_users = (
    conversions.groupby("user_id")
    .size()
    .gt(1)
    .sum()
)

print("Users with multiple conversions:",
      multi_conversion_users)

Users with multiple conversions: 0


### Create the User Journey Table

In [26]:
# Join touchpoints with conversion info

journey = touchpoints.merge(
    conversions[
        ["user_id",
         "conversion_id",
         "conversion_timestamp",
         "revenue"]
    ],
    on="user_id",
    how="inner"
)

journey.shape

(27250, 15)

In [27]:
journey = journey[
    journey["timestamp"] <=
    journey["conversion_timestamp"]
]

journey.shape

(27189, 15)

### Journey Sequence

In [29]:
journey = journey.sort_values(
    ["user_id", "timestamp"]
)

journey["touch_order"] = (
    journey.groupby("user_id")
    .cumcount() + 1
)

journey.head(20)

,user_id,session_id,timestamp,source,medium,campaign_id,campaign_name,event_type,landing_page,device_type,country,city,conversion_id,conversion_timestamp,revenue,touch_order
14580,USR0000001,SES88091408,2025-06-14 16:44:15,google,cpc,CMP0001,Google_Brand_Search,download,/blog/marketing-tips,desktop,FR,Marseille,CNV0007274,2025-06-18 10:19:15,1467.27,1
14581,USR0000001,SES93836866,2025-06-14 16:51:15,facebook,paid_social,CMP0013,Meta_Lead_Gen_SaaS,form_submit,/about,desktop,FR,Marseille,CNV0007274,2025-06-18 10:19:15,1467.27,2
14582,USR0000001,SES94460736,2025-06-15 17:56:15,google,cpc,CMP0002,Google_Competitor_Search,sign_up,/contact,desktop,FR,Marseille,CNV0007274,2025-06-18 10:19:15,1467.27,3
14583,USR0000001,SES59349415,2025-06-17 05:17:15,linkedin,paid_social,CMP0022,LinkedIn_Retargeting_B2B,page_view,/landing/b2b-demo,desktop,FR,Marseille,CNV0007274,2025-06-18 10:19:15,1467.27,4
17403,USR0000002,SES10800646,2025-02-25 22:16:33,google,cpc,CMP0006,Google_Shopping_Fashion,download,/landing/b2b-demo,desktop,US,New York,CNV0005904,2025-02-28 05:55:33,796.39,1
17404,USR0000002,SES72830790,2025-02-26 06:03:33,facebook,paid_social,CMP0012,Meta_Carousel_Products,form_submit,/electronics,desktop,US,New York,CNV0005904,2025-02-28 05:55:33,796.39,2
26331,USR0000004,SES31218045,2025-01-03 15:18:08,google,cpc,CMP0004,Google_Display_Retargeting,page_view,/home-goods,mobile,FR,Paris,CNV0003610,2025-01-07 20:06:08,843.83,1
26332,USR0000004,SES48819480,2025-01-04 02:22:08,linkedin,paid_social,CMP0022,LinkedIn_Retargeting_B2B,download,/free-trial,mobile,FR,Paris,CNV0003610,2025-01-07 20:06:08,843.83,2
26333,USR0000004,SES66495113,2025-01-04 13:23:08,google,cpc,CMP0008,Google_PMax_Conversions,add_to_cart,/signup,mobile,FR,Paris,CNV0003610,2025-01-07 20:06:08,843.83,3
26334,USR0000004,SES36793296,2025-01-05 02:29:08,linkedin,paid_social,CMP0021,LinkedIn_Brand_Awareness_B2B,add_to_cart,homepage,mobile,FR,Paris,CNV0003610,2025-01-07 20:06:08,843.83,4
